# IOAI — 2025 Lab Lab5 Liars Dice (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
!pip -q install python-decouple langchain-google-genai langchain-core
import urllib.request
urllib.request.urlretrieve('https://raw.githubusercontent.com/jaredliw/ioai-tsp-2025/main/lab5/liarsdice.py','liarsdice.py')
open('.env','a').close()
print('⚠️ Gemini API 키가 필요합니다(.env 에 GOOGLE_API_KEY=...). liarsdice.py 준비 완료.')
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# Lab 5 Part 2 - Liar's Dice

In [ ]:
# 추가 의존성 설치 + API 키 안내
!pip install -q python-decouple langchain-google-genai langchain-core
import os
open('.env','a').close()  # .env 없으면 RepositoryEnv 가 실패하므로 빈 파일 보장
print('⚠️ Gemini API 키가 필요합니다: .env 에 GOOGLE_API_KEY=... (aistudio.google.com)')


In Part 1, you are introduced to orchestrating LLM calls using Langchain. In Part 2, we will use that knowledge to pit LLMs against each other to play a full game of Liar's Dice.

This video succintly explains Liar's Dice: https://www.youtube.com/watch?v=fAbnMuiR734.

Before we go any further, I will first show you a game transcript using the sample solution.

## Sample transcript

```
LIAR'S DICE!
Players: Alice, Bob
Total dice: 6

DICE (visible for demo):
Alice: [5, 4, 1]
Bob: [6, 4, 1]

                       --- TURN 1 ---                       

============================================================
ROUND 1 - GAME STATE
============================================================
PLAYERS:
   Alice: [5, 4, 1] (3 dice) ← CURRENT TURN
   Bob: [6, 4, 1] (3 dice)

CURRENT BID: None (round start)

📊 TOTAL DICE IN PLAY: 6

Alice's turn

🤔 Alice is thinking...
💭 Alice's thoughts:
   Okay, let's analyze this Liar's Dice situation. I have [5, 4, 1]. The game has just started, so I'm the first bidder. There are a total of 6 dice in play.

1. My hand: I have a 5, a 4, and a 1. The 1 is wild. That means I have at least one "5" and one "4" (or a 1 acting as either). I have no other matching dice besides the 1.

2. Opponent inference: Since it's the first bid, I have no information about my opponents' dice.

3. Mathematical odds: I can't calculate odds without a bid. However, I know the distribution of dice is essentially random at the start. With 6 dice, the expected number of any specific face (including 1s) is around 1.0. I have a 1, so I have a better than average chance of seeing 1's in the total.

4. Bid or challenge: Since I'm the first bidder, I can't challenge. I must bid.

5. Strategic bid: Given my hand, I want to start with a bid that is low, but not too low. I'll use the 1 as a "5" for the bid. I could bid "one 5". However, I have a 5 and a 1 so the odds of there being more than that is pretty decent. I'm going to bid "two 5s". This is a reasonable starting point that hopefully gives me a chance to get some information.

🤖 Alice decides: BID 2 5
✅ Alice bids 2 dice show 5

                       --- TURN 2 ---                       

============================================================
ROUND 1 - GAME STATE
============================================================
PLAYERS:
   Alice: [5, 4, 1] (3 dice)
   Bob: [6, 4, 1] (3 dice) ← CURRENT TURN

CURRENT BID: 2 dice show 5 (by Alice)

📊 TOTAL DICE IN PLAY: 6

📜 RECENT HISTORY:
   • Alice bids 2 dice show 5

Bob's turn

🤔 Bob is thinking...
💭 Bob's thoughts:
   Okay, let's analyze this Liar's Dice situation.

1. My hand: I have a 6, a 4, and a 1. The 1 is a wild, so I essentially have two dice that don't help me with the current bid of "2 dice showing 5".

2. Opponent's bidding pattern: Alice bid "2 dice show 5". Since I don't know anything else, it doesn't tell me anything about her hand. She could have a 5 and a wild, or she could be bluffing.

3. Mathematical odds: There are 6 dice in total. The bid is 2 dice showing 5. To calculate the probability, we need to consider the odds of rolling 5s on 6 dice.

*   **Zero 5s:** (4/6)^6
*   **One 5:** 6 * (1/6) * (4/6)^5
*   **Two 5s:** 15 * (1/6)^2 * (4/6)^4

If we use the 1s as wilds, the probabilities are a bit harder, but the chance of there being two 5s or more is pretty high.

4. Should I bid higher or challenge? Given my hand, I can't really *help* the bid, and the odds might favor me. I think I should challenge.

5. Strategic considerations: A challenge now is the best course of action. I have a very bad hand for this bid, and I don't want to start bidding on something I don't have a good chance of getting.

🤖 Bob decides: CHALLENGE

                  🚨 CHALLENGE RESOLUTION 🚨                  
Bob challenges Alice's bid:
BID: 2 dice show 5

COUNTING DICE SHOWING 5:
   Alice: 1 natural 5s + 1 wilds = 2 total
   Bob: 0 natural 5s + 1 wilds = 1 total

FINAL COUNT: 3 dice show 5
BID CLAIMED: 2 dice show 5
✅ BID WAS TRUE! Bob loses a die!
Alice wins the challenge!
Bob now has 2 dice

Rolling new dice for next round...
   Alice: [6, 5, 4]
   Bob: [2, 6]

NEW ROUND 2! Alice starts with fresh dice

                       --- TURN 3 ---                       

============================================================
ROUND 2 - GAME STATE
============================================================
PLAYERS:
   Alice: [6, 5, 4] (3 dice) ← CURRENT TURN
   Bob: [2, 6] (2 dice)

CURRENT BID: None (round start)

📊 TOTAL DICE IN PLAY: 5

📜 RECENT HISTORY:
   • Alice bids 2 dice show 5
   • Bob challenges Alice's bid
   • Challenge: Bob vs Alice - Alice wins

Alice's turn

🤔 Alice is thinking...
💭 Alice's thoughts:
   Okay, let's analyze this Liar's Dice situation.

First, let's recall the game history: I lost the last round because I bid 2 fives, and Bob challenged. It appears Bob is challenging frequently or he has a good hand.

Now, let's assess my hand: [6, 5, 4]. I have one five.

The current bid is: None. It is my turn to bid and I start.

I know the total number of dice is 5. The previous bid was 2 fives by me, which got challenged and was successful.

Considering my hand, I have one five. With the total dice count at 5, the minimum bid I should consider is 1 five. If I include the possibility that my other opponent also has a five, I can consider 2 fives.

I will start by bidding 1 five.

My bid: 1 die show 5.

🤖 Alice decides: BID 1 5
✅ Alice bids 1 dice show 5

                       --- TURN 4 ---                       

============================================================
ROUND 2 - GAME STATE
============================================================
PLAYERS:
   Alice: [6, 5, 4] (3 dice)
   Bob: [2, 6] (2 dice) ← CURRENT TURN

CURRENT BID: 1 dice show 5 (by Alice)

📊 TOTAL DICE IN PLAY: 5

📜 RECENT HISTORY:
   • Alice bids 2 dice show 5
   • Bob challenges Alice's bid
   • Challenge: Bob vs Alice - Alice wins
   • Alice bids 1 dice show 5

Bob's turn

🤔 Bob is thinking...
💭 Bob's thoughts:
   Okay, let's analyze this Liar's Dice situation.

1. My hand: I have a 2 and a 6. The current bid is 1 die showing a 5. I have no 5s, and only two dice. This makes it slightly more likely the bid is false.

2. Opponent inference: Alice bid 2 fives, and won the challenge. Then she bid 1 five. This suggests she is confident about having 5s, or is trying to bluff. The first bid was higher than the current, so she could have had more 5s. She probably has at least one die that shows a 5.

3. Mathematical odds: There are 5 dice total. The bid is for one 5. Considering 1s are wild, the bid is a 1/6 chance of being true on any given die, plus the wild, which makes it likely. Alice has a die, so it could be true. Without the wild, the chance is 1 - (5/6)^5 = 0.598. With a wild, it increases significantly, but is still less than 1. Because the previous bid was for two 5s, and the current bid is for one, I need to be careful.

4. Bid or Challenge? Given the opponent's bidding pattern, and the possible probabilities, I think it is best to challenge. If I bid, I might be forced to bid even higher, which is less likely given the low chance of the bid being true. Challenging is the better move this round.

5. No bid.

🤖 Bob decides: CHALLENGE

                  🚨 CHALLENGE RESOLUTION 🚨                  
Bob challenges Alice's bid:
BID: 1 dice show 5

COUNTING DICE SHOWING 5:
   Alice: 1 natural 5s + 0 wilds = 1 total
   Bob: 0 natural 5s + 0 wilds = 0 total

FINAL COUNT: 1 dice show 5
BID CLAIMED: 1 dice show 5
✅ BID WAS TRUE! Bob loses a die!
Alice wins the challenge!
Bob now has 1 dice

Rolling new dice for next round...
   Alice: [2, 5, 2]
   Bob: [3]

NEW ROUND 3! Alice starts with fresh dice

                       --- TURN 5 ---                       

============================================================
ROUND 3 - GAME STATE
============================================================
PLAYERS:
   Alice: [2, 5, 2] (3 dice) ← CURRENT TURN
   Bob: [3] (1 dice)

CURRENT BID: None (round start)

📊 TOTAL DICE IN PLAY: 4

📜 RECENT HISTORY:
   • Bob challenges Alice's bid
   • Challenge: Bob vs Alice - Alice wins
   • Alice bids 1 dice show 5
   • Bob challenges Alice's bid
   • Challenge: Bob vs Alice - Alice wins

Alice's turn

🤔 Alice is thinking...
💭 Alice's thoughts:
   Okay, let's analyze this Liar's Dice situation.

My dice: [2, 5, 2]
Total dice: 4
Current bid: None
Game History: Alice wins the previous two challenges.

1. My hand: I have two 2s and one 5. This gives me a decent starting hand. The fact that I have only three dice in play means I want to be conservative with my bids.

2. Opponent's inference: Bob has challenged twice and lost twice. This means Bob either thinks my bids are lies, or is just bad at estimating.

3. Mathematical odds: I need to consider the previous bids. I won the last two challenges. Now I need to determine the best bid.

4. Should I bid or challenge? Since I start the round I need to bid. I have to determine a good bid.

5. Strategic bid: I know there are only 4 dice in the game. Since I have two 2s, I could bid:
* "2 dice show 2"
* "2 dice show 5"
* "3 dice show 2"
* "1 dice show 5"
* "1 dice show 1" (Wild)

Given the history, I should bid lower. My strategy is to bid "1 dice show 5". This is a safe bid. It is possible based on my hand and the game history.

🤖 Alice decides: BID 1 5
✅ Alice bids 1 dice show 5

                       --- TURN 6 ---                       

============================================================
ROUND 3 - GAME STATE
============================================================
PLAYERS:
   Alice: [2, 5, 2] (3 dice)
   Bob: [3] (1 dice) ← CURRENT TURN

CURRENT BID: 1 dice show 5 (by Alice)

📊 TOTAL DICE IN PLAY: 4

📜 RECENT HISTORY:
   • Challenge: Bob vs Alice - Alice wins
   • Alice bids 1 dice show 5
   • Bob challenges Alice's bid
   • Challenge: Bob vs Alice - Alice wins
   • Alice bids 1 dice show 5

Bob's turn

🤔 Bob is thinking...
💭 Bob's thoughts:
   Okay, let's analyze this Liar's Dice situation.

1. My hand: I have one die showing a 3. The current bid is 1 die showing a 5. Given my hand, it's impossible for me to contribute to the bid, but I can't count out the possibility.

2. Opponent's Bidding: Alice has been successful twice in a row. Each time, her bids have been for a quantity of 1 die showing 5. This means she could have a 5, a 1, or she might be bluffing.

3. Odds: There are 4 dice in the game. The current bid is 1 die showing 5. The probability of at least one die showing a 5 (or a 1, which counts as a 5) is moderately high. It's probably higher than 50%.

4. Decision: I have one die. Given my hand, bidding higher is not a good idea. Challenging is risky because I don't have a strong reason to believe she's lying. Given Alice's pattern, and the number of dice, I think she's probably telling the truth, or the chances of her lying are close enough to 50/50 to justify a challenge. Let's challenge.

5. Action: Challenge

🤖 Bob decides: CHALLENGE

                  🚨 CHALLENGE RESOLUTION 🚨                  
Bob challenges Alice's bid:
BID: 1 dice show 5

COUNTING DICE SHOWING 5:
   Alice: 1 natural 5s + 0 wilds = 1 total
   Bob: 0 natural 5s + 0 wilds = 0 total

FINAL COUNT: 1 dice show 5
BID CLAIMED: 1 dice show 5
✅ BID WAS TRUE! Bob loses a die!
Alice wins the challenge!
Bob now has 0 dice
Bob is eliminated! 💀

🎉 Alice WINS THE GAME! 🎉

Game finished after 5 turns!
Final game history:
   1. Alice bids 2 dice show 5
   2. Bob challenges Alice's bid
   3. Challenge: Bob vs Alice - Alice wins
   4. Alice bids 1 dice show 5
   5. Bob challenges Alice's bid
   6. Challenge: Bob vs Alice - Alice wins
   7. Alice bids 1 dice show 5
   8. Bob challenges Alice's bid
   9. Challenge: Bob vs Alice - Alice wins
```

## Setup

In [ ]:
from decouple import Config, RepositoryEnv

In [ ]:
# TODO: 여기에 풀이를 작성하세요


In [ ]:
# TODO: 여기에 풀이를 작성하세요


In [ ]:
# TODO: 여기에 풀이를 작성하세요


In [ ]:
config = Config(RepositoryEnv(".env"))

In [ ]:
GOOGLE_API_KEY = config("GOOGLE_API_KEY")

## Making a functional Liar's Dice game

A game orchestrator has already been implemented for you in `liarsdice.py`. The only missing part to start your first game of Liar's Dice is to implement `AIPlayer` below. Study the orchestrator to understand how the game is run before filling out the code below. Make any changes you want anywhere, as long as you can get it to work.

In [ ]:
# TODO: 여기에 풀이를 작성하세요


When you are done, try running a game of Liar's Dice below!

In [ ]:
# TODO: 여기에 풀이를 작성하세요


## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv', 'submission.zip', 'submission.jsonl', 'submission.json']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)